In [ ]:
pip install pycaret

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 4.5 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of category-encoders to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of pmdarima to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 486.1/486.1 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.8/106.8 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.8/21.8 MB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.4/85.4 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.2/302.2 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 5.7 M

In [ ]:
from google.colab import drive
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import joblib
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.metrics import accuracy_score, precision_score, confusion_matrix, classification_report, ConfusionMatrixDisplay
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from pycaret.classification import setup,compare_models,tune_model,evaluate_model,predict_model,finalize_model,save_model,ensemble_model

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [ ]:
drive.mount('/content/drive')

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Aprendizaje de Maquinas/Obesity.csv', header = 0)

In [ ]:
df

# Data Cleaning
- Verificación de valores nulos
- Codificación de variables categóricas


In [ ]:
# Nulos
print(df.isnull().sum())

In [ ]:
categorical_cols = df.select_dtypes(include='object').columns
print("Columnas categóricas:", categorical_cols.tolist())

In [ ]:
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)
df_encoded

##Análisis del dataset

El dataset contiene información relacionada con hábitos alimenticios, actividad física y condición física de diferentes individuos. Está compuesto por variables numéricas y categóricas, las cuales permiten dar una estimación de los niveles de obesidad de individuos de Peru, México y Colombia.

Las preguntas realizadas fueron:

*   **family_history**: ¿Algún miembro de tu familia ha sufrido o sufre de sobrepeso?
*   **FAVC**: ¿Consumes alimentos de alto contenido calórico con frecuencia?
*   **FCVC**: ¿Sueles comer verduras en tus comidas?
*   **NCP**: ¿Cuántas comidas principales consumes al día?
*   **CAEC**: ¿Comes algo entre comidas?
*   **SMOKE**: ¿Fumas?
*   **CH2O**: ¿Cuánta agua bebes al día?
*   **SCC**: ¿Controlas las calorías que consumes diariamente?
*   **FAF**: ¿Con qué frecuencia realizas actividad física?
*   **TUE**: ¿Cuánto tiempo utilizas dispositivos tecnológicos como el celular, videojuegos, televisión, computadora y otros?
*   **CALC**: ¿Con qué frecuencia consumes alcohol?
*   **MTRANS**: ¿Qué medio de transporte utilizas habitualmente?

In [ ]:
print(df.describe())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

sns.boxplot(x='Obesity', y='Age', data=df, ax=ax, palette='viridis', hue='Obesity', legend=False)
ax.set_title('Distribución de Edad por Nivel de Obesidad')
ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
fig, ax= plt.subplots(figsize=(10, 7))

sns.boxplot(x='Obesity', y='Height', data=df, ax=ax, palette='plasma', hue='Obesity', legend=False)
ax.set_title('Distribución de Altura por Nivel de Obesidad')
ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

sns.boxplot(x='Obesity', y='Weight', data=df, ax=ax, palette='cividis', hue='Obesity', legend=False)
ax.set_title('Distribución de Peso por Nivel de Obesidad')
ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

Se observa una amplia variabilidad en variables como el peso y la edad. Además, la altura presenta una menor dispersión respecto al peso, por lo que tenemos una mayor consistencia entre los registros.

Aunque las clases mantienen una distribución relativamente equilibrada, algunas presentan una mayor cantidad de registros.

In [ ]:
print("Distribución según nivel de obesidad:")
sns.countplot(x='Obesity', data=df)
plt.xticks(rotation=45)
plt.show()

In [ ]:
print(f"El DataFrame tiene {df.shape[0]} filas y {df.shape[1]} columnas.")
print(f"Tipos de datos del dataframe {df.dtypes}")

##Análisis de correlación

In [ ]:
corr = df.corr(numeric_only=True)

plt.figure(figsize=(10,6))
sns.heatmap(corr, annot=True, cmap='coolwarm')
plt.show()

El peso y la altura son las variables numéricas más interrelacionadas, y la mayoría de las otras variables numéricas actúan de manera relativamente independiente entre sí en términos de correlación lineal.

La variable dependiente es Obesity ya que es la variable que el modelo intentará predecir. En este caso la codificamos con LabelEncoder.

In [ ]:
y = df['Obesity']
le = LabelEncoder()
y_encoded = le.fit_transform(y)

In [ ]:
joblib.dump(le, 'label_encoder.pkl')

Las variables independientes son todas las demás columnas en el DataFrame que llamamos X.

In [ ]:
df_encoded['IMC'] = df_encoded['Weight'] / (df_encoded['Height']**2)
display(df_encoded.head())

In [ ]:
obesity_cols_encoded = [col for col in df_encoded.columns if 'Obesity_' in col]
X = df_encoded.drop(columns=obesity_cols_encoded + ['Height', 'Weight'])

X.head()

Conjuntos de entrenamiento, pruebas y validación

In [ ]:
X_temp, X_val, y_temp, y_val = train_test_split(X, y_encoded, test_size=0.05, random_state=3, stratify=y_encoded)
X_train, X_test, y_train, y_test = train_test_split(X_temp, y_temp, test_size=0.15, random_state=3, stratify=y_temp)

print(f'Dimensiones del Conjunto de Entrenamiento: {X_train.shape[0]} muestras')
print(f'Dimensiones del Conjunto de Validación: {X_val.shape[0]} muestras')
print(f'Dimensiones del Conjunto de Prueba: {X_test.shape[0]} muestras')

print(f'\nProporción del Conjunto de Entrenamiento: {X_train.shape[0]/len(X)*100:.2f}%')
print(f'Proporción del Conjunto de Validación: {X_val.shape[0]/len(X)*100:.2f}%')
print(f'Proporción del Conjunto de Prueba: {X_test.shape[0]/len(X)*100:.2f}%')

##Ingeniería de características

Decidimos incorporar el Índice de Masa Corporal (IMC) como una nueva característica.
Esta combina la información de altura y peso en un solo valor que clasifica el estado de peso (bajo peso, normal, sobrepeso, obeso) y es una métrica fundamental que nos ayuda mucho más a la predicción de nivel de obesidad.

## Entrenamiento del Modelo Random Forest

In [ ]:
print(X_train.columns)
X_train.head(1)

In [ ]:
rf_model = RandomForestClassifier(n_estimators=200, max_depth=12, random_state=3)
rf_model.fit(X_train, y_train)

## Evaluación del Modelo

In [ ]:
y_pred = rf_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f'Exactitud (Accuracy): {accuracy:.4f}')

In [ ]:
precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
print(f'Precisión: {precision:.4f}')

In [ ]:
mc = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(mc, annot=True, fmt='d', cmap='Blues', xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('Matriz de Confusión')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
class_report = classification_report(y_test, y_pred, target_names=le.classes_, zero_division=0)
print('\nReporte de Clasificación:')
print(class_report)

In [ ]:
joblib.dump(rf_model, 'RandomForest_model.pkl')

##Implementacion Pipeline

In [ ]:
def add_imc_and_delete(X):
    X = X.copy()
    X['IMC'] = X['Weight'] / (X['Height'] ** 2)
    X = X.drop(columns=['Height', 'Weight'])
    return X

imc_transformer = FunctionTransformer(add_imc_and_delete)

In [ ]:
numerical_cols = ['Age', 'FCVC', 'NCP', 'CH2O', 'FAF', 'TUE', 'IMC']
categorical_cols = ['Gender', 'family_history', 'FAVC', 'CAEC', 'SMOKE', 'SCC', 'CALC', 'MTRANS']
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numerical_cols),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_cols)
])

In [ ]:
pipeline = Pipeline(steps=[
    ('imc',   imc_transformer),
    ('prep',  preprocessor),
    ('modelo', RandomForestClassifier(
                  n_estimators=200,
                  max_depth=12,
                  random_state=3
              ))
])

In [ ]:
X_temp, X_val, y_temp, y_val = train_test_split(df.drop(columns=['Obesity']), y_encoded, test_size=0.05, random_state=3, stratify=y_encoded)
X_train, X_test, y_train, y_test = train_test_split(X_temp, y_temp, test_size=0.15, random_state=3, stratify=y_temp)
pipeline.fit(X_train, y_train)

In [ ]:
y_pred = pipeline.predict(X_test)

class_report = classification_report(y_test, y_pred, target_names=le.classes_, zero_division=0)
print('\nReporte de Clasificación:')
print(class_report)

In [ ]:
joblib.dump(pipeline, 'pipeline_completa.pkl')

##Implementacion PyCaret

In [ ]:
s = setup(data=X_train,
                target='Obesity',
                session_id=3)

In [ ]:
best = compare_models()

In [ ]:
tunned_model = tune_model(best)

##Métodos de ensamble

In [ ]:
bagged_model = ensemble_model(best, method='Bagging')

In [ ]:
boosted_model = ensemble_model(best, method='Boosting')

Se evidencia que el bagged model es mejor que el modelo base y el boosted, asi que se trabajara con el

In [ ]:
best = bagged_model

In [ ]:
evaluate_model(best)

In [ ]:
predict_model(best)

In [ ]:
predict_model(best, data=X_test)

In [ ]:
finalize_model(best)

In [ ]:
save_model(best, 'pycaret_model')